# Can we prove ACLED's cursor pagination returns the same data as offset?

**Yes — and this notebook is the audit trail, not the proof.** The proof is
the committed JSON it reads. Nothing here contacts ACLED.

> **Evidence as of 2026-10-12.** This notebook is a *record of a measurement
> taken at a point in time*, not a check that re-runs. It cannot be re-run
> against the live API after **27 October 2026**, because on that date the
> offset method ceases to exist and one half of the comparison becomes
> impossible to obtain. That is the whole reason it was taken early.

**How to read it.** Work down. Every number printed below is read from a file
in `reports/investigation_acled_cursor/` or `tests/fixtures/acled_cursor/`,
all of which are committed. Where this notebook computes something itself, it
says so, and it recomputes rather than reprints — section 5 re-derives the
content digest from the raw recorded responses, so you are not asked to trust
the script that produced the report.

**What would make this notebook a lie.** If the two walks had each silently
stopped early at the same page, they would agree on every data oracle while
both being wrong. That failure mode is real — it was found in this
investigation's own instrument during review — so a fifth oracle,
`walks_ran_to_completion`, is reported alongside the four data ones, and a
capped walk forces a non-parity verdict. Check it in section 3.

## 1. Why this measurement exists

ACLED's own notice, relayed verbatim in two independent third-party trackers:

> "We've rolled out a new cursor-based pagination method and will deprecate
> the current offset method on **27 October 2026**."

- [mapme.biodiversity#492](https://github.com/mapme-initiative/mapme.biodiversity/issues/492)
- [forecastbench#315](https://github.com/forecastingresearch/forecastbench/issues/315)

Neither is ACLED's own page, which is why the field name and placement were
confirmed against a live response before any code was written (section 2).

Our harvester walked ACLED by `&page=N`. After 27 October that parameter
stops working.

| Date | What |
|---|---|
| 29–30 Sep 2026 | The notice appears in downstream trackers |
| **21 Oct 2026** | Monthly pipeline cron — last run where offset still works |
| **27 Oct 2026** | Offset removed by ACLED |
| **21 Nov 2026** | First run that **must** be on cursor |

**The failure shape is why this was urgent rather than routine.** A missed
migration does not crash on the 27th. It surfaces on 21 November as a short
harvest — quiet, a month late, and in the dataset rather than the logs. This
repository has already been burned by exactly that shape: ADR-027 exists
because `page_size=50000` made the UCDP API silently return 335,918 of
384,918 events while the harvest reported success.

In [1]:
"""Load the committed evidence. No network access anywhere in this notebook."""

import hashlib
import json
from pathlib import Path

REPORTS = Path("..") / "reports" / "investigation_acled_cursor"
FIXTURES = Path("..") / "tests" / "fixtures" / "acled_cursor"

# Three slices were measured. They answer different questions, which is why
# all three are kept rather than only the largest.
SLICES = {
    "quarter": "parity_quarter.json",
    "week": "parity.json",
    "fixture": "parity_fixture_slice.json",
}

reports = {
    name: json.loads((REPORTS / fname).read_text())
    for name, fname in SLICES.items()
}

for name, rep in reports.items():
    sl = rep["slice"]
    print(f"{name:8s}  {sl['event_date']}  limit={sl['limit']:<5d} "
          f"country={sl['country']:<8s} "
          f"measured {rep['measured_at_utc'][:19]}Z  at {rep['git_sha']}")

quarter   2024-01-01|2024-03-31  limit=5000  country=all      measured 2026-10-12T01:35:08Z  at 70164d3
week      2024-03-01|2024-03-07  limit=1000  country=all      measured 2026-10-10T23:11:30Z  at 47089fd
fixture   2024-03-01|2024-03-07  limit=50    country=Somalia  measured 2026-10-10T23:13:15Z  at 47089fd


## 2. What was confirmed against the live API first

Three things were established with one authenticated request **before** any
implementation code was written, because the plan depended on all three and
the public notice is not an API contract:

| Question | Answer, observed |
|---|---|
| Is `next_cursor` a top-level key in the JSON body? | Yes |
| What type is it? | **`int`** — e.g. `1148657`, monotonically increasing |
| Do `count` / `total_count` carry values now? | **No, still `null`** |

**The type mattered more than it looks.** The walk starts at `cursor=0`, and
`0` is both the legitimate starting value and falsy in Python. Terminating on
`if not next_cursor` would therefore have been a live bug that passed every
happy-path test. Termination is on `next_cursor is None`, and a mutant that
swaps it for a falsiness check is in the mutation set specifically because
this observation suggested it.

The `count`/`total_count` answer is why risk-register entry C-153 was
**amended rather than closed**: there is still no server-declared total to
validate a harvest against, so ADR-027's tolerance rule remains inapplicable
to ACLED.

## 3. The two walks, side by side

Each slice was fetched **twice in the same session, minutes apart** — once by
`&page=N` using a frozen copy of the pre-migration production loop, once by
`&cursor=X`. Same session matters: ACLED restates history continuously, so
two fetches separated by months would differ for reasons that have nothing to
do with pagination.

In [2]:
hdr = (f"{'slice':9s}{'events (offset)':>17s}{'events (cursor)':>17s}"
       f"{'reqs':>6s}{'reqs':>6s}{'completed':>11s}")
print(hdr)
print("-" * len(hdr))

for name, rep in reports.items():
    completed = rep["checks"]["walks_ran_to_completion"]
    capped = rep["offset_hit_request_cap"] or rep["cursor_hit_request_cap"]
    print(f"{name:9s}{rep['offset_n_events']:>17,d}"
          f"{rep['cursor_n_events']:>17,d}"
          f"{rep['offset_requests']:>6d}{rep['cursor_requests']:>6d}"
          f"{str(completed and not capped):>11s}")

slice      events (offset)  events (cursor)  reqs  reqs  completed
------------------------------------------------------------------
quarter             98,636           98,636    20    20       True
week                 7,609            7,609     8     8       True
fixture                 84               84     2     2       True


### The five oracles

Parity requires **every one** of these to hold. Four are about the data; the
fifth is about the measurement itself.

| Oracle | What it would catch |
|---|---|
| `count_equal` | A dropped or duplicated page |
| `id_sets_equal` | Different events, same count |
| `records_equal` | Same events, a differing field value |
| `digests_equal` | The ledger would record a different value |
| `walks_ran_to_completion` | **Both walks truncating at the request cap** |

The digest is load-bearing because it is the exact value the ingestion ledger
records and `_year_is_cached()` compares against — so agreement here means
agreement on the thing the pipeline actually persists. It is also computed
over *sorted* records, which is why a different page ordering between the two
methods could not produce a false mismatch.

Full-record equality is asserted alongside it because the digest covers only
`event_id_cnty` and `fatalities`, and would not have seen a divergence in,
say, `latitude`.

In [3]:
oracles = ["count_equal", "id_sets_equal", "records_equal",
           "digests_equal", "walks_ran_to_completion"]

for name, rep in reports.items():
    results = " ".join(
        f"{o}={'PASS' if rep['checks'][o] else 'FAIL'}" for o in oracles
    )
    print(f"{name:9s} {results}")
    print(f"{'':9s} digest offset={rep['offset_content_digest']} "
          f"cursor={rep['cursor_content_digest']}")
    print(f"{'':9s} only_in_offset={rep['only_in_offset']} "
          f"only_in_cursor={rep['only_in_cursor']}")
    print(f"{'':9s} PARITY: {rep['parity']}\n")

quarter   count_equal=PASS id_sets_equal=PASS records_equal=PASS digests_equal=PASS walks_ran_to_completion=PASS
          digest offset=af9fb35c03ac830f cursor=af9fb35c03ac830f
          only_in_offset=[] only_in_cursor=[]
          PARITY: True

week      count_equal=PASS id_sets_equal=PASS records_equal=PASS digests_equal=PASS walks_ran_to_completion=PASS
          digest offset=ae4a6e815168e95f cursor=ae4a6e815168e95f
          only_in_offset=[] only_in_cursor=[]
          PARITY: True

fixture   count_equal=PASS id_sets_equal=PASS records_equal=PASS digests_equal=PASS walks_ran_to_completion=PASS
          digest offset=5b73a9160f8a8256 cursor=5b73a9160f8a8256
          only_in_offset=[] only_in_cursor=[]
          PARITY: True



## 4. The widest slice ran at the page size production actually uses

The first measurement used `limit=1000`. Production harvests with
`page_size=5000` (`AcledConfig`), so for a while no measurement had exercised
the page size the pipeline really runs.

That is not a cosmetic gap. `limit` is the exact parameter the **deleted**
`len(results) < config.page_size` terminator compared against, so changing it
changes the arithmetic of the condition this migration removed.

In [4]:
q = reports["quarter"]
w = reports["week"]
ratio = q["cursor_n_events"] / w["cursor_n_events"]

print(f"week    : {w['cursor_n_events']:>7,d} events  "
      f"limit={w['slice']['limit']:<5d} {w['cursor_requests']:>3d} requests")
print(f"quarter : {q['cursor_n_events']:>7,d} events  "
      f"limit={q['slice']['limit']:<5d} {q['cursor_requests']:>3d} requests")
print(f"\nthe quarter slice carries {ratio:.1f}x the events of the week slice")
print(f"and is the only one measured at production's page_size "
      f"({q['slice']['limit']})")

week    :   7,609 events  limit=1000    8 requests
quarter :  98,636 events  limit=5000   20 requests

the quarter slice carries 13.0x the events of the week slice
and is the only one measured at production's page_size (5000)


## 5. Recomputing the digest from the raw recorded responses

Everything above is read from a report file. This section does not trust those
reports: it loads the **raw HTTP response envelopes** recorded during the
narrow run and re-derives the content digest independently, using the same
definition as `event_validation.py` — sort the `(event_id_cnty, fatalities)`
tuples, serialise with sorted keys, SHA-256, keep 16 hex characters.

If this recomputation disagrees with the recorded digest, the report is wrong.

**These envelopes are the irreplaceable artifact.** After 27 October the
offset side of this recording can never be produced again by anyone.

In [5]:
DIGEST_FIELDS = ("event_id_cnty", "fatalities")  # acled.py


def content_digest(events: list[dict]) -> str:
    """Re-derivation of validate_events(...).content_digest."""
    rows = sorted(
        tuple(ev.get(f, 0) for f in DIGEST_FIELDS) for ev in events
    )
    payload = json.dumps(rows, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()[:16]


walks = {
    name: json.loads((FIXTURES / f"{name}_walk.json").read_text())
    for name in ("offset", "cursor")
}

for name, pages in walks.items():
    print(f"{name} walk: {len(pages)} recorded response(s)")
    for i, page in enumerate(pages, start=1):
        cursor = page.get("next_cursor", "<absent: offset envelope>")
        print(f"  page {i}: {len(page['data']):>3d} events  "
              f"next_cursor={cursor}")

offset walk: 2 recorded response(s)
  page 1:  50 events  next_cursor=<absent: offset envelope>
  page 2:  34 events  next_cursor=<absent: offset envelope>
cursor walk: 2 recorded response(s)
  page 1:  50 events  next_cursor=1992089
  page 2:  34 events  next_cursor=None


In [6]:
recorded = reports["fixture"]["cursor_content_digest"]

print(f"{'walk':8s}{'events':>8s}  {'recomputed digest':20s}"
      f"{'matches report':>16s}")
print("-" * 54)
for name, pages in walks.items():
    events = [ev for page in pages for ev in page["data"]]
    digest = content_digest(events)
    print(f"{name:8s}{len(events):>8d}  {digest:20s}"
          f"{str(digest == recorded):>16s}")

offset_events = [ev for p in walks["offset"] for ev in p["data"]]
cursor_events = [ev for p in walks["cursor"] for ev in p["data"]]
offset_ids = {ev["event_id_cnty"] for ev in offset_events}
cursor_ids = {ev["event_id_cnty"] for ev in cursor_events}

def by_id(events: list[dict]) -> list[dict]:
    return sorted(events, key=lambda e: e["event_id_cnty"])


print(f"\nreport recorded          : {recorded}")
print(f"symmetric difference     : {offset_ids ^ cursor_ids or '(empty)'}")
print(f"full records equal       : "
      f"{by_id(offset_events) == by_id(cursor_events)}")

walk      events  recomputed digest     matches report
------------------------------------------------------
offset        84  5b73a9160f8a8256                True
cursor        84  5b73a9160f8a8256                True

report recorded          : 5b73a9160f8a8256
symmetric difference     : (empty)
full records equal       : True


Note what the page structure shows about termination — and note carefully
what it does **not** show.

The final page carries `next_cursor: null`, and the walk stops there because
the server said so, not because the page looked short. Under cursor pagination
that is the only authoritative terminator: a short page mid-series is not the
end, and the old code's `len(results) < limit` inference would have stopped
early on one.

**But the final page was short in every slice measured** — 34 of 50 here,
609 of 1,000 in the week slice, 3,636 of 5,000 in the quarter. So the case
where the old code is *unambiguously* wrong, a final page that is exactly
full, never occurred live. The old code would have terminated correctly on
all three of these slices.

That case is covered only by `test_full_final_page_with_null_cursor_terminates`,
which constructs it synthetically. It is a real limitation of this evidence
and it is the same limitation as the negative H4 result in section 6: the
measurement confirms the two methods agree, and does *not* demonstrate the
new terminator succeeding where the old one would have failed.

## 6. What this notebook does **not** establish

Stated plainly, because an evidence document that only lists its strengths is
advocacy.

1. **One hypothesis came back negative.** No page ever arrived short *while
   the cursor was still live*, across any slice. So the deleted short-page
   break would have agreed with the new code on everything measured here. The
   deletion is structurally correct — a short page genuinely is not the end of
   a cursor series — but it is **not vindicated by this data**. The honest
   basis for it is that restoring the break is caught by the mutation drill,
   not that the measurement caught it.

2. **The decisive case was never observed live.** The final page was short
   in all three slices, so the one configuration where the deleted terminator
   is unambiguously wrong -- a final page containing exactly `limit` events,
   followed by `next_cursor: null` -- was never produced by ACLED during the
   measurement. It is covered by a synthetic unit test only. This is the same
   gap as point 1, seen from the other side.

3. **Parity on three slices is not parity on all history.** The slices are
   2024, all countries, no event-type filter. Nothing here measures behaviour
   across an ACLED restatement, or at the year boundaries the production
   harvester actually walks.

4. **A short harvest would still be recorded as a success.** `n_events` goes
   into the ingestion ledger and nothing reads it back. This migration makes
   the *loop* unable to truncate silently; it does not make the *ledger* able
   to notice if something else does. Tracked as C-359 and issue #488, and
   deliberately not fixed under deadline.

5. **This notebook is not a test.** CI does not execute it. The claim it
   renders is enforced by `tests/test_acled_cursor_parity_fixture.py`, which
   replays the same recorded envelopes through the real `fetch_paginated()`
   and asserts the event count, page count and digest. The notebook is the
   human-readable argument *about* that test, not a substitute for it.

## 7. Verdict

In [7]:
all_parity = all(rep["parity"] for rep in reports.values())
total_events = sum(rep["cursor_n_events"] for rep in reports.values())
no_caps = all(
    not (rep["offset_hit_request_cap"] or rep["cursor_hit_request_cap"])
    for rep in reports.values()
)

print("ACLED cursor-vs-offset pagination parity")
print("=" * 48)
print(f"slices measured          : {len(reports)}")
print(f"events compared in total : {total_events:,d}")
print(f"every oracle held        : {all_parity}")
print(f"no walk hit its cap      : {no_caps}")
print()
if all_parity and no_caps:
    print("At the timestamps and against the slices recorded above, ACLED's")
    print("cursor pagination returned exactly the same events as its offset")
    print("pagination -- identical counts, identical event IDs, identical")
    print("field values, and an identical content digest, on walks that each")
    print("ran to the server's own declared end of series.")
else:
    print("NOT PARITY -- do not read this notebook as supporting the change.")

ACLED cursor-vs-offset pagination parity
slices measured          : 3
events compared in total : 106,329
every oracle held        : True
no walk hit its cap      : True

At the timestamps and against the slices recorded above, ACLED's
cursor pagination returned exactly the same events as its offset
pagination -- identical counts, identical event IDs, identical
field values, and an identical content digest, on walks that each
ran to the server's own declared end of series.
